# 🎮 GTA 6 Viral Clipper — Kaggle Edition
Gere cortes verticais virais automáticos para TikTok, Reels e Shorts direto no **Kaggle**!

### ⚡ Vantagens no Kaggle:
- **30 horas semanais de GPU Grátis** (NVIDIA T4 x2 ou P100).
- **30 GB de RAM** e internet ultrarrápida.
- **Execução em segundo plano:** Você pode dar *Save Version -> Save & Run*, fechar o navegador e o Kaggle continua gerando os cortes por até 12 horas!

> ⚠️ **PASSO OBRIGATÓRIO (Configuração no Kaggle):**
> Na barra lateral direita do notebook (**Notebook settings**):
> 1. **Accelerator:** selecione **GPU T4 x2** (ou GPU P100)
> 2. **Internet:** marque **Internet on** (obrigatório para baixar vídeos e dependências)


In [ ]:
# ⚙️ 1. Instalar Dependências & Clonar Repositório
%cd /kaggle/working
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-montserrat
!pip install -q -U --pre "yt-dlp[default]"
!pip install -q faster-whisper opencv-python-headless numpy certifi static-ffmpeg deno

# Baixar modelo facial YuNet para Split-Screen automático
!mkdir -p /root/clipper-gta6/modelos
!curl -fsSL -o /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx

# Clonar projeto atualizado do GitHub
!rm -rf /kaggle/working/meu-gta6-clipper
!git clone -q --depth 1 https://github.com/papaigato/gta6-auto-clips.git /kaggle/working/meu-gta6-clipper
%cd /kaggle/working/meu-gta6-clipper
print('✓ Ambiente Kaggle configurado com sucesso!')

In [ ]:
# 🎬 2. Configurar Live e Gerar Cortes Automáticos
import os, glob, sys

# ================= PARÂMETROS DA LIVE =================
LINK_DA_LIVE = "https://youtu.be/Q9ZqVAdtASE"
QUANTIDADE_DE_CORTES = 4
RESOLUCAO_SAIDA = "Full HD (1080x1920)"  # Opções: '2K (1440x2560)' ou 'Full HD (1080x1920)'
FPS_SAIDA = "60 fps"                    # Opções: '60 fps' ou '30 fps'
RESOLUCAO_DOWNLOAD = "1080p60"          # Opções: '1440p60', '1080p60', '720p60'
ESTILO_VISUAL = "split"                 # Opções: 'split' (webcam + jogo), 'blur' (fundo desfocado) ou 'preencher'
CORTES_EM_PARALELO = 2
REFAZER_CORTES = True
# ======================================================

PASTA_DESTINO = "/kaggle/working/GTA6_Cortes"
CACHE_DIR = "/kaggle/working/.cache"
TMP_DIR = "/tmp/clipper_gta6"

os.makedirs(PASTA_DESTINO, exist_ok=True)
os.makedirs(f"{CACHE_DIR}/modelos", exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

if os.path.exists("/root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx"):
    !cp -u /root/clipper-gta6/modelos/face_detection_yunet_2023mar.onnx "$CACHE_DIR/modelos/"

max_res = "1440" if "1440" in RESOLUCAO_DOWNLOAD else ("1080" if "1080" in RESOLUCAO_DOWNLOAD else "720")
out_res = "2k" if "2K" in RESOLUCAO_SAIDA else "1080p"
fps_val = "30" if "30" in FPS_SAIDA else "60"

os.environ["CLIPPER_MAX_RES"] = max_res
os.environ["CLIPPER_OUT_RES"] = out_res
os.environ["CLIPPER_FPS"] = fps_val
os.environ["CLIPPER_PARALELO"] = str(CORTES_EM_PARALELO)
os.environ["CLIPPER_CORTES_DIR"] = PASTA_DESTINO
os.environ["CLIPPER_GTA6_HOME"] = CACHE_DIR
os.environ["CLIPPER_TMP_DIR"] = TMP_DIR
os.environ["CLIPPER_ESTILO"] = ESTILO_VISUAL
os.environ["CLIPPER_FORCAR_REFAZER"] = "1" if REFAZER_CORTES else "0"
os.environ["CLIPPER_ESPERA"] = "86400"

print(f"🚀 Iniciando Clipper no Kaggle!")
print(f"📺 Live: {LINK_DA_LIVE}")
print(f"💎 Saída: {out_res.upper()} @ {fps_val}fps | Download: {max_res}p | Estilo: {ESTILO_VISUAL}")

%cd /kaggle/working/meu-gta6-clipper
!python /kaggle/working/meu-gta6-clipper/scripts/clip.py automatico "$LINK_DA_LIVE" --quantos $QUANTIDADE_DE_CORTES

In [ ]:
# 📦 3. Compactar Cortes em ZIP para Baixar
import glob, os, shutil
from IPython.display import FileLink, display

videos = glob.glob(f"{PASTA_DESTINO}/*/*.mp4")
print(f"🎬 Total de vídeos prontos: {len(videos)}")
for v in sorted(videos):
    mb = os.path.getsize(v) / (1024 * 1024)
    print(f"  ✓ {os.path.basename(v)} ({mb:.1f} MB)")

zip_path = "/kaggle/working/gta6_cortes_prontos"
shutil.make_archive(zip_path, "zip", PASTA_DESTINO)
zip_file = f"{zip_path}.zip"

if os.path.exists(zip_file):
    tamanho_mb = os.path.getsize(zip_file) / (1024 * 1024)
    print(f"\n✅ Arquivo ZIP gerado: {zip_file} ({tamanho_mb:.1f} MB)")
    print("👉 Clique no link abaixo para baixar ou use o painel Output lateral direito:")
    display(FileLink(os.path.relpath(zip_file, "/kaggle/working")))

In [ ]:
# 👁️ 4. (Opcional) Assistir a um dos Cortes no Navegador
from IPython.display import Video, display
import glob

videos = sorted(glob.glob(f"{PASTA_DESTINO}/*/*.mp4"))
if videos:
    print(f"Reproduzindo: {videos[0]}")
    display(Video(videos[0], embed=True, width=320))
else:
    print("Nenhum corte encontrado para reproduzir.")